# Book Recommendation Model

## Imports and data loading

In [1]:
## Imports and data loading

import numpy as np
import pandas as pd
import warnings
from pathlib import Path
warnings.filterwarnings("ignore")

DATA_DIR = Path("books_data")

books = pd.read_csv(
    DATA_DIR / "books.csv",
    sep=";",
    encoding="latin-1",
    on_bad_lines="skip",
    engine="python"
)

ratings = pd.read_csv(
    DATA_DIR / "ratings.csv",
    sep=";",
    encoding="latin-1",
    on_bad_lines="skip",
    engine="python"
)

users = pd.read_csv(
    DATA_DIR / "users.csv",
    sep=";",
    encoding="latin-1",
    on_bad_lines="skip",
    engine="python"
)

## Data Cleaning (Might need changes)

In [ ]:

books_clean = books.copy()
ratings_clean = ratings.copy()

books_clean.columns = books_clean.columns.str.strip()
ratings_clean.columns = ratings_clean.columns.str.strip()

books_clean["ISBN"] = books_clean["ISBN"].astype(str).str.strip()
ratings_clean["ISBN"] = ratings_clean["ISBN"].astype(str).str.strip()

ratings_clean["User-ID"] = pd.to_numeric(ratings_clean["User-ID"], errors="coerce")
ratings_clean["Book-Rating"] = pd.to_numeric(ratings_clean["Book-Rating"], errors="coerce")

ratings_clean = ratings_clean.dropna(subset=["User-ID", "ISBN", "Book-Rating"])
ratings_clean["User-ID"] = ratings_clean["User-ID"].astype(int)
ratings_clean["Book-Rating"] = ratings_clean["Book-Rating"].astype(int)

# For this step, rating 0 still counts as "read/interacted".
# We are NOT saying rating 0 means liked.
read_events = ratings_clean[["User-ID", "ISBN"]].drop_duplicates()

# Keep only books that exist in the books table
read_events = read_events[read_events["ISBN"].isin(books_clean["ISBN"])].copy()

books_small = books_clean[["ISBN", "Book-Title", "Book-Author", "Year-Of-Publication", "Publisher"]].drop_duplicates("ISBN")

print("Read events:", len(read_events))
print("Unique users:", read_events["User-ID"].nunique())
print("Unique books:", read_events["ISBN"].nunique())

Read events: 1028716
Unique users: 91990
Unique books: 269288


## Book ISBN search based on title for future actions

In [ ]:
def search_books_by_title(title, n=10):
    """
    Search books by title so you can find the correct ISBN.
    """
    title = title.lower()

    matches = books_small[books_small["Book-Title"].astype(str).str.lower().str.contains(title, na=False)].copy()

    return matches.head(n)

In [4]:
search_books_by_title("flies", n=5) 

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher
1755,0399501487,Lord of the Flies,William Gerald Golding,1959,Perigee Trade
1839,0385240406,Time Flies,Bill Cosby,1987,Bantam Dell Pub Group
3469,0373263678,Death Flies On Final (Wwl Mystery),Jackie Lewin,2000,Worldwide Library
5656,0553277243,Time Flies,BILL COSBY,1988,Bantam
14339,0452274427,In the Time of the Butterflies,Julia Alvarez,1995,Plume Books


## Co-reader vote recommender

In [ ]:
def recommend_by_coreader_votes(seed_isbns, n_recommendations=10):
    """
    Recommend books using simple co-reader voting.

    Logic:
    1)Input books are books the user READ.
    2)Find users who also read those input books.
    3)Every other book read by those users gets votes.
    4)If a user matched 2 input books, their other books get +2.
    5)Return the books with the highest vote scores.
    """

    #Clean input ISBNs
    seed_isbns = [str(isbn).strip() for isbn in seed_isbns]
    seed_isbns = list(set(seed_isbns))

    #Check which input ISBNs exist in the books table
    valid_seed_isbns = [isbn for isbn in seed_isbns if isbn in set(books_small["ISBN"])]

    print("Input seed ISBNs:", seed_isbns)
    print("Valid seed ISBNs found:", valid_seed_isbns)
    print("Number of valid seed ISBNs:", len(valid_seed_isbns))

    if len(valid_seed_isbns) == 0:
        raise ValueError("None of the input ISBNs were found in the books table.")

    #Find users who read at least one of the seed books
    seed_reads = read_events[read_events["ISBN"].isin(valid_seed_isbns)].copy()

    if seed_reads.empty:
        raise ValueError("No users read the input seed books.")

    #Count how many seed books each user matched
    user_seed_matches = (
        seed_reads
        .groupby("User-ID")["ISBN"]
        .nunique()
        .rename("seed_match_count")
        .reset_index()
    )

    #Get all books read by those matching users
    candidate_reads = read_events.merge(user_seed_matches, on="User-ID", how="inner")

    #Do not recommend the input books back
    candidate_reads = candidate_reads[
        ~candidate_reads["ISBN"].isin(valid_seed_isbns)
    ].copy()

    if candidate_reads.empty:
        return pd.DataFrame()

    #Score candidates
    scores = (
        candidate_reads
        .groupby("ISBN")
        .agg(
            coreader_vote_score=("seed_match_count", "sum"),
            matched_reader_count=("User-ID", "nunique")
        )
        .reset_index()
    )

    #Global reader count: how many users in the whole data read this book
    global_book_counts = (
        read_events
        .groupby("ISBN")["User-ID"]
        .nunique()
        .rename("global_reader_count")
        .reset_index()
    )

    scores = scores.merge(global_book_counts, on="ISBN", how="left")

    #Adjusted score reduces pure popularity bias
    scores["adjusted_score"] = (scores["coreader_vote_score"] / np.sqrt(scores["global_reader_count"]))

    #Sort by raw co-reader votes
    scores = scores.sort_values(by=["coreader_vote_score", "matched_reader_count"], ascending=False)
    recommendations = scores.head(n_recommendations).merge( books_small, on="ISBN", how="left")

    return recommendations

## Test

In [6]:
seed_isbns = [
    "0451139712",   # The Stand
    "0451157443",   # Carrie
    "0743424425	",  # The Shining
    "0451184963",   # Insomnia
    "044021145X",   # The Firm
    "0345337662",   # Interview with the Vampire
    "0440224675",   # Hannibal
    "0399501487",   # Lord of the Flies
    "0671027360",   # Angels and Demons

]

recommend_by_coreader_votes(
    seed_isbns,
    n_recommendations=20
)

Input seed ISBNs: ['0451184963', '0399501487', '0743424425', '0451139712', '0671027360', '0440224675', '0345337662', '0451157443', '044021145X']
Valid seed ISBNs found: ['0451184963', '0399501487', '0743424425', '0451139712', '0671027360', '0440224675', '0345337662', '0451157443', '044021145X']
Number of valid seed ISBNs: 9


,ISBN,coreader_vote_score,matched_reader_count,global_reader_count,adjusted_score,Book-Title,Book-Author,Year-Of-Publication,Publisher
0,0971880107,590,384,2502,11.795283,Wild Animus,Rich Shapero,2004,Too Far
1,0440214041,505,279,523,22.082108,The Pelican Brief,John Grisham,1993,Dell
2,0440211727,484,256,517,21.286296,A Time to Kill,JOHN GRISHAM,1992,Dell
3,0316666343,482,292,1295,13.394057,The Lovely Bones: A Novel,Alice Sebold,2002,"Little, Brown"
4,0385504209,461,284,883,15.513885,The Da Vinci Code,Dan Brown,2003,Doubleday
5,0345370775,409,216,466,18.946559,Jurassic Park,Michael Crichton,1999,Ballantine Books
6,0060928336,405,230,732,14.969231,Divine Secrets of the Ya-Ya Sisterhood: A Novel,Rebecca Wells,1997,Perennial
7,067976402X,372,206,614,15.012698,Snow Falling on Cedars,David Guterson,1995,Vintage Books USA
8,0345313860,371,205,301,21.384084,"The Vampire Lestat (Vampire Chronicles, Book II)",ANNE RICE,1986,Ballantine Books
9,0804106304,363,196,519,15.933932,The Joy Luck Club,Amy Tan,1994,Prentice Hall (K-12)


In [12]:
def test_random_user_recommendations(
    min_books_read=15,
    n_seed_books=15,
    n_recommendations=20,
    random_state=42
):
    # Count how many books each user has read/interacted with
    user_book_counts = read_events.groupby("User-ID")["ISBN"].nunique()

    # Keep only users with at least min_books_read books
    eligible_users = user_book_counts[user_book_counts >= min_books_read].index

    print("Eligible users:", len(eligible_users))

    if len(eligible_users) == 0:
        raise ValueError("No users found with enough read books.")

    # Pick one random user
    random_user_id = pd.Series(eligible_users).sample(n=1, random_state=random_state).iloc[0]

    print("Random user:", random_user_id)

    # Get all books read by this user
    random_user_books = read_events[ read_events["User-ID"] == random_user_id].copy()

    # Add book information to all read books
    random_user_books_info = random_user_books.merge( books_small, on="ISBN", how="left")

    # Pick n_seed_books random books from this user as input
    seed_books = random_user_books.sample( n=n_seed_books, random_state=random_state).copy()

    seed_isbns = seed_books["ISBN"].tolist()

    # Add book information to seed books
    seed_books_info = seed_books.merge( books_small, on="ISBN", how="left")

    print("\nSeed ISBNs:")
    print(seed_isbns)
    print("\nBooks used as input:")
    display(seed_books_info)

    # Run the beta recommendation model
    recommendations = recommend_by_coreader_votes( seed_isbns, n_recommendations=n_recommendations)

    print("\nRecommendations:")
    display(recommendations)

    # Check overlap between user's read books and recommendations
    read_isbns = set(random_user_books["ISBN"])
    recommended_isbns = set(recommendations["ISBN"])

    overlap_isbns = read_isbns.intersection(recommended_isbns)
    overlap_books = recommendations[recommendations["ISBN"].isin(overlap_isbns)].copy()

    print("\nOverlap check:")
    print("Total books read by user:", len(read_isbns))
    print("Total recommendations:", len(recommended_isbns))
    print("Recommended books already read by user:", len(overlap_isbns))

    if len(recommended_isbns) > 0:
        print(
            "Share of recommendations already read:",
            round(len(overlap_isbns) / len(recommended_isbns), 3)
        )

    print("\nRecommended books that the user had already read:")
    display(overlap_books)

    return random_user_id, seed_books_info, recommendations, random_user_books_info, overlap_books


random_user_id, seed_books_info, recommendations, random_user_books_info, overlap_books = test_random_user_recommendations(
    min_books_read=30,
    n_seed_books=20,
    n_recommendations=10,
    random_state=42
)

Eligible users: 4733
Random user: 137118

Seed ISBNs:
['055356918X', '0425179885', '0380795310', '0553572164', '0345339703', '0940601109', '0505524244', '0590344188', '0671004565', '0345339738', '0553576445', '0399139710', '0451406869', '0380805685', '0446609463', '0688037852', '0451197186', '0789458586', '0717283364', '0061057517']

Books used as input:


,User-ID,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher
0,137118,055356918X,Twice a Hero,Susan Krinard,1997,Bantam Books
1,137118,0425179885,The Devil's Code,John Sandford,2004,Berkley Publishing Group
2,137118,0380795310,Walk This Way : The Autobiography of Aerosmith,Aerosmith,1999,Avon
3,137118,0553572164,F2F,Phillip Finch,1997,Bantam
4,137118,0345339703,The Fellowship of the Ring (The Lord of the Ri...,J.R.R. TOLKIEN,1986,Del Rey
5,137118,0940601109,Dad and Son: A Memoir About Reclaiming Fatherh...,Arthur C. Klein,1996,Champion Pr Ltd
6,137118,0505524244,Shadower (Futuristic Romance),Catherine Spangler,2000,Love Spell
7,137118,0590344188,Men in Black: A Novelization,J. J. Gardner,1997,Scholastic Paperbacks (Mm)
8,137118,0671004565,We'll Meet Again,Mary Higgins Clark,2000,Pocket
9,137118,0345339738,"The Return of the King (The Lord of the Rings,...",J.R.R. TOLKIEN,1986,Del Rey


Input seed ISBNs: ['0345339703', '0505524244', '0688037852', '0717283364', '0380795310', '055356918X', '0590344188', '0451197186', '0553572164', '0940601109', '0345339738', '0553576445', '0446609463', '0671004565', '0789458586', '0061057517', '0399139710', '0451406869', '0380805685', '0425179885']
Valid seed ISBNs found: ['0345339703', '0505524244', '0688037852', '0717283364', '0380795310', '055356918X', '0590344188', '0451197186', '0553572164', '0940601109', '0345339738', '0553576445', '0446609463', '0671004565', '0789458586', '0061057517', '0399139710', '0451406869', '0380805685', '0425179885']
Number of valid seed ISBNs: 20

Recommendations:


,ISBN,coreader_vote_score,matched_reader_count,global_reader_count,adjusted_score,Book-Title,Book-Author,Year-Of-Publication,Publisher
0,0345339711,229,129,177,17.212694,"The Two Towers (The Lord of the Rings, Part 2)",J.R.R. TOLKIEN,1986,Del Rey
1,0345339681,156,84,281,9.306180,The Hobbit : The Enchanting Prelude to The Lor...,J.R.R. TOLKIEN,1986,Del Rey
2,0971880107,155,115,2502,3.098761,Wild Animus,Rich Shapero,2004,Too Far
3,0440214041,147,99,523,6.427861,The Pelican Brief,John Grisham,1993,Dell
4,0440211727,146,99,517,6.421073,A Time to Kill,JOHN GRISHAM,1992,Dell
5,044021145X,124,79,529,5.391304,The Firm,John Grisham,1992,Bantam Dell Publishing Group
6,0316666343,118,88,1295,3.279043,The Lovely Bones: A Novel,Alice Sebold,2002,"Little, Brown"
7,059035342X,118,87,571,4.938146,Harry Potter and the Sorcerer's Stone (Harry P...,J. K. Rowling,1999,Arthur A. Levine Books
8,0385504209,111,78,883,3.735447,The Da Vinci Code,Dan Brown,2003,Doubleday
9,0440220602,105,65,379,5.393488,The Chamber,John Grisham,1995,Dell Publishing Company



Overlap check:
Total books read by user: 42
Total recommendations: 10
Recommended books already read by user: 2
Share of recommendations already read: 0.2

Recommended books that the user had already read:


,ISBN,coreader_vote_score,matched_reader_count,global_reader_count,adjusted_score,Book-Title,Book-Author,Year-Of-Publication,Publisher
0,0345339711,229,129,177,17.212694,"The Two Towers (The Lord of the Rings, Part 2)",J.R.R. TOLKIEN,1986,Del Rey
1,0345339681,156,84,281,9.306180,The Hobbit : The Enchanting Prelude to The Lor...,J.R.R. TOLKIEN,1986,Del Rey
